# 📊 AI Resume & Job Matcher — Complete EDA + Model Analysis
### By Abhishek Ainapure | B.Tech Artificial Intelligence
---
**This notebook covers:**
- Step 1: Import Libraries
- Step 2: Load Dataset
- Step 3: Data Understanding
- Step 4: Data Cleaning
- Step 5: Exploratory Data Analysis (EDA)
- Step 6: Feature Engineering
- Step 7: Model Building (KNN + Random Forest)
- Step 8: Model Evaluation (Accuracy, F1, Confusion Matrix)
- Step 9: Model Comparison
- Step 10: Insights & Conclusion

## Step 1 — Import All Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('✅ All libraries imported successfully!')

## Step 2 — Load Dataset

In [ ]:
# Load jobs dataset
# If you have downloaded Kaggle dataset, replace the path below
# df_jobs = pd.read_csv('../data/job_descriptions.csv')

# Using sample dataset for demonstration
df_jobs = pd.read_csv('../data/jobs.csv')

print(f'✅ Jobs Dataset Loaded!')
print(f'Shape: {df_jobs.shape}')
print(f'Rows: {df_jobs.shape[0]}, Columns: {df_jobs.shape[1]}')

In [ ]:
# Preview dataset
df_jobs.head(10)

## Step 3 — Data Understanding

In [ ]:
# Basic info
print('=== DATASET INFO ===')
print(df_jobs.info())

In [ ]:
# Column names
print('Columns:', df_jobs.columns.tolist())

In [ ]:
# Statistical summary
print('=== STATISTICAL SUMMARY ===')
df_jobs.describe(include='all')

In [ ]:
# Check missing values
print('=== MISSING VALUES ===')
missing = df_jobs.isnull().sum()
missing_pct = (df_jobs.isnull().sum() / len(df_jobs)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)
print(f'\nTotal missing values: {df_jobs.isnull().sum().sum()}')

In [ ]:
# Check duplicates
print(f'Duplicate rows: {df_jobs.duplicated().sum()}')
print(f'Unique job titles: {df_jobs["job_title"].nunique()}')
print(f'Unique companies: {df_jobs["company"].nunique()}')
print(f'Unique job roles: {df_jobs["job_role"].nunique()}')
print(f'\nJob Roles: {df_jobs["job_role"].unique()}')
print(f'Experience Levels: {df_jobs["experience_level"].unique()}')

## Step 4 — Data Cleaning

In [ ]:
# Make a clean copy
df = df_jobs.copy()

# Fill missing values
df.fillna('unknown', inplace=True)

# Convert text columns to lowercase
for col in ['job_title', 'company', 'required_skills', 'experience_level', 'job_role']:
    if col in df.columns:
        df[col] = df[col].str.lower().str.strip()

# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f'Rows before cleaning: {before}')
print(f'Rows after cleaning: {after}')
print(f'Duplicates removed: {before - after}')
print('\n✅ Data Cleaning Done!')

In [ ]:
# Add skill_count column
df['skill_count'] = df['required_skills'].apply(lambda x: len(str(x).split(',')))

# Encode experience level
exp_map = {'fresher': 0, 'mid': 1, 'senior': 2, 'unknown': 0}
df['exp_encoded'] = df['experience_level'].map(exp_map).fillna(0)

print('New columns added:')
print(df[['job_title', 'skill_count', 'exp_encoded']].head())

## Step 5 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Plot 1: Job Role Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
role_counts = df['job_role'].value_counts()
axes[0].bar(role_counts.index, role_counts.values, color=['#6c63ff','#43e97b','#ff6584'])
axes[0].set_title('Job Role Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Job Role')
axes[0].set_ylabel('Count')
for i, v in enumerate(role_counts.values):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(role_counts.values, labels=role_counts.index, autopct='%1.1f%%',
            colors=['#6c63ff','#43e97b','#ff6584'], startangle=140)
axes[1].set_title('Job Role Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/plot1_job_roles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Most jobs in dataset belong to Data Science domain')

In [ ]:
# ── Plot 2: Experience Level Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

exp_counts = df['experience_level'].value_counts()

axes[0].bar(exp_counts.index, exp_counts.values,
            color=['#43e97b','#6c63ff','#ff6584'])
axes[0].set_title('Experience Level Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Experience Level')
axes[0].set_ylabel('Count')
for i, v in enumerate(exp_counts.values):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

axes[1].pie(exp_counts.values, labels=exp_counts.index, autopct='%1.1f%%',
            colors=['#43e97b','#6c63ff','#ff6584'])
axes[1].set_title('Experience Level Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/plot2_experience.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Dataset is balanced across fresher, mid, senior levels')

In [ ]:
# ── Plot 3: Skill Count Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['skill_count'], bins=10, color='#6c63ff', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Required Skills per Job', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Required Skills')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['skill_count'].mean(), color='#ff6584', linestyle='--',
                label=f'Mean = {df["skill_count"].mean():.1f}')
axes[0].legend()

# Box plot by role
roles = df['job_role'].unique()
data_by_role = [df[df['job_role']==r]['skill_count'].values for r in roles]
axes[1].boxplot(data_by_role, labels=roles, patch_artist=True,
                boxprops=dict(facecolor='#6c63ff', alpha=0.6))
axes[1].set_title('Skill Count by Job Role', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Job Role')
axes[1].set_ylabel('Skill Count')

plt.tight_layout()
plt.savefig('../data/plot3_skills.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Average required skills per job: {df["skill_count"].mean():.1f}')

In [ ]:
# ── Plot 4: Top Skills in Job Market ──
all_skills = []
for skills_str in df['required_skills']:
    skills = [s.strip() for s in str(skills_str).split(',')]
    all_skills.extend(skills)

from collections import Counter
skill_counts = Counter(all_skills).most_common(15)
skills_df = pd.DataFrame(skill_counts, columns=['Skill', 'Count'])

plt.figure(figsize=(12, 6))
bars = plt.barh(skills_df['Skill'], skills_df['Count'],
                color=plt.cm.viridis(np.linspace(0.2, 0.9, len(skills_df))))
plt.title('Top 15 Most In-Demand Skills', fontsize=14, fontweight='bold')
plt.xlabel('Number of Job Postings')
plt.ylabel('Skill')
for bar, count in zip(bars, skills_df['Count']):
    plt.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
             str(count), va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plot4_top_skills.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Most demanded skill: {skill_counts[0][0]}')

In [ ]:
# ── Plot 5: Skills Heatmap by Role ──
top_skills = [s[0] for s in skill_counts[:10]]
roles = df['job_role'].unique()

heatmap_data = []
for role in roles:
    role_skills = []
    role_df = df[df['job_role'] == role]
    for skill in top_skills:
        count = role_df['required_skills'].str.contains(skill, case=False, na=False).sum()
        role_skills.append(count)
    heatmap_data.append(role_skills)

heatmap_df = pd.DataFrame(heatmap_data, index=roles, columns=top_skills)

plt.figure(figsize=(14, 5))
sns.heatmap(heatmap_df, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Count'})
plt.title('Skill Demand Heatmap by Job Role', fontsize=14, fontweight='bold')
plt.xlabel('Skills')
plt.ylabel('Job Role')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/plot5_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Heatmap shows which skills are required for each job role')

In [ ]:
# ── Plot 6: Correlation Analysis ──
df['role_encoded'] = LabelEncoder().fit_transform(df['job_role'])

corr_df = df[['skill_count', 'exp_encoded', 'role_encoded']].copy()
corr_matrix = corr_df.corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, mask=mask, linewidths=1,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plot6_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Correlation shows relationships between features')

## Step 6 — Feature Engineering

In [ ]:
# Create synthetic training data for model evaluation
# In real project this comes from actual resume-job match data

np.random.seed(42)
n_samples = 200

training_data = pd.DataFrame({
    'skill_count': np.random.randint(2, 15, n_samples),
    'experience_years': np.random.randint(0, 8, n_samples),
    'education_level': np.random.randint(0, 3, n_samples),
    'keyword_match': np.random.uniform(20, 95, n_samples).round(1),
})

# Generate job role based on skill count (realistic rule)
def assign_role(row):
    if row['skill_count'] >= 7 and row['keyword_match'] >= 60:
        return 'data science'
    elif row['skill_count'] >= 5 and row['keyword_match'] >= 50:
        return 'data science'
    else:
        return 'web dev'

def assign_exp_level(years):
    if years == 0: return 'fresher'
    elif years <= 2: return 'mid'
    else: return 'senior'

training_data['job_role'] = training_data.apply(assign_role, axis=1)
training_data['experience_level'] = training_data['experience_years'].apply(assign_exp_level)
training_data['ats_score'] = (
    training_data['skill_count'] * 3.5 +
    training_data['keyword_match'] * 0.4 +
    training_data['experience_years'] * 2 +
    training_data['education_level'] * 3
).clip(0, 100).round(1)

print('Training Data Shape:', training_data.shape)
print('\nSample:')
print(training_data.head())
print('\nJob Role Distribution:')
print(training_data['job_role'].value_counts())
print('\nExperience Level Distribution:')
print(training_data['experience_level'].value_counts())

In [ ]:
# Prepare features and targets
feature_cols = ['skill_count', 'experience_years', 'education_level', 'keyword_match']
X = training_data[feature_cols]

# Encode targets
le_role = LabelEncoder()
le_exp = LabelEncoder()

y_role = le_role.fit_transform(training_data['job_role'])
y_exp = le_exp.fit_transform(training_data['experience_level'])
y_ats = training_data['ats_score']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Features:', feature_cols)
print('X shape:', X.shape)
print('Role classes:', le_role.classes_)
print('Experience classes:', le_exp.classes_)

## Step 7 — Model Building

In [ ]:
# Train-Test Split
X_train, X_test, y_train_role, y_test_role = train_test_split(
    X_scaled, y_role, test_size=0.2, random_state=42, stratify=y_role
)
_, _, y_train_exp, y_test_exp = train_test_split(
    X_scaled, y_exp, test_size=0.2, random_state=42, stratify=y_exp
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples: {X_test.shape[0]}')
print(f'Split ratio: 80% train, 20% test')

In [ ]:
# ── Model 1: KNN ──
print('=== KNN MODEL ===')

# Find best K
k_values = range(1, 21)
k_scores = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    score = cross_val_score(knn, X_train, y_train_role, cv=5, scoring='accuracy').mean()
    k_scores.append(score)

best_k = k_values[np.argmax(k_scores)]
print(f'Best K value: {best_k}')
print(f'Best CV Accuracy: {max(k_scores):.4f}')

# Plot K vs Accuracy
plt.figure(figsize=(10, 5))
plt.plot(k_values, k_scores, 'b-o', linewidth=2, markersize=6)
plt.axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.title('KNN: K Value vs Cross-Validation Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('K Value')
plt.ylabel('CV Accuracy')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('../data/plot7_knn_k.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Train final KNN model
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train, y_train_role)
knn_pred = knn_model.predict(X_test)

knn_accuracy = accuracy_score(y_test_role, knn_pred)
knn_f1 = f1_score(y_test_role, knn_pred, average='weighted')
knn_precision = precision_score(y_test_role, knn_pred, average='weighted')
knn_recall = recall_score(y_test_role, knn_pred, average='weighted')

print(f'KNN Accuracy:  {knn_accuracy:.4f} ({knn_accuracy*100:.2f}%)')
print(f'KNN F1 Score:  {knn_f1:.4f}')
print(f'KNN Precision: {knn_precision:.4f}')
print(f'KNN Recall:    {knn_recall:.4f}')
print('\nClassification Report:')
print(classification_report(y_test_role, knn_pred, target_names=le_role.classes_))

In [ ]:
# ── Model 2: Random Forest ──
print('=== RANDOM FOREST MODEL ===')

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf_model.fit(X_train, y_train_role)
rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test_role, rf_pred)
rf_f1 = f1_score(y_test_role, rf_pred, average='weighted')
rf_precision = precision_score(y_test_role, rf_pred, average='weighted')
rf_recall = recall_score(y_test_role, rf_pred, average='weighted')

print(f'RF Accuracy:  {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)')
print(f'RF F1 Score:  {rf_f1:.4f}')
print(f'RF Precision: {rf_precision:.4f}')
print(f'RF Recall:    {rf_recall:.4f}')
print('\nClassification Report:')
print(classification_report(y_test_role, rf_pred, target_names=le_role.classes_))

In [ ]:
# ── Model 3: Logistic Regression (for comparison) ──
print('=== LOGISTIC REGRESSION (Comparison) ===')

lr_model = LogisticRegression(random_state=42, max_iter=500)
lr_model.fit(X_train, y_train_role)
lr_pred = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test_role, lr_pred)
lr_f1 = f1_score(y_test_role, lr_pred, average='weighted')
lr_precision = precision_score(y_test_role, lr_pred, average='weighted')
lr_recall = recall_score(y_test_role, lr_pred, average='weighted')

print(f'LR Accuracy:  {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)')
print(f'LR F1 Score:  {lr_f1:.4f}')
print(f'LR Precision: {lr_precision:.4f}')
print(f'LR Recall:    {lr_recall:.4f}')

## Step 8 — Model Evaluation

In [ ]:
# ── Confusion Matrices ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_preds = [
    ('KNN', knn_pred),
    ('Random Forest', rf_pred),
    ('Logistic Regression', lr_pred)
]

for ax, (name, pred) in zip(axes, models_preds):
    cm = confusion_matrix(y_test_role, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=le_role.classes_,
                yticklabels=le_role.classes_)
    ax.set_title(f'{name}\nConfusion Matrix', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../data/plot8_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Diagonal values show correct predictions')

In [ ]:
# ── Cross Validation Scores ──
print('=== 5-FOLD CROSS VALIDATION ===')

cv_results = {}
for name, model in [('KNN', knn_model), ('Random Forest', rf_model), ('Logistic Regression', lr_model)]:
    scores = cross_val_score(model, X_scaled, y_role, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f'{name}:')
    print(f'  Scores: {scores.round(4)}')
    print(f'  Mean: {scores.mean():.4f} | Std: {scores.std():.4f}')
    print()

In [ ]:
# ── Cross Validation Box Plot ──
plt.figure(figsize=(10, 6))
cv_data = [cv_results['KNN'], cv_results['Random Forest'], cv_results['Logistic Regression']]
bp = plt.boxplot(cv_data, labels=['KNN', 'Random Forest', 'Logistic Regression'],
                  patch_artist=True, notch=False)
colors = ['#6c63ff', '#43e97b', '#ff6584']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
plt.title('Cross-Validation Score Distribution', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy')
plt.xlabel('Model')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('../data/plot9_cv_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Model Comparison

In [ ]:
# ── Complete Model Comparison Table ──
comparison_df = pd.DataFrame({
    'Model': ['KNN', 'Random Forest', 'Logistic Regression'],
    'Accuracy': [knn_accuracy, rf_accuracy, lr_accuracy],
    'F1 Score': [knn_f1, rf_f1, lr_f1],
    'Precision': [knn_precision, rf_precision, lr_precision],
    'Recall': [knn_recall, rf_recall, lr_recall],
    'CV Mean': [
        cv_results['KNN'].mean(),
        cv_results['Random Forest'].mean(),
        cv_results['Logistic Regression'].mean()
    ]
})

# Format as percentage
for col in ['Accuracy','F1 Score','Precision','Recall','CV Mean']:
    comparison_df[col] = (comparison_df[col] * 100).round(2).astype(str) + '%'

print('=== FINAL MODEL COMPARISON ===')
print(comparison_df.to_string(index=False))

In [ ]:
# ── Model Comparison Bar Chart ──
metrics = ['Accuracy', 'F1 Score', 'Precision', 'Recall']
knn_vals = [knn_accuracy, knn_f1, knn_precision, knn_recall]
rf_vals = [rf_accuracy, rf_f1, rf_precision, rf_recall]
lr_vals = [lr_accuracy, lr_f1, lr_precision, lr_recall]

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, knn_vals, width, label='KNN', color='#6c63ff', alpha=0.85)
bars2 = ax.bar(x, rf_vals, width, label='Random Forest', color='#43e97b', alpha=0.85)
bars3 = ax.bar(x + width, lr_vals, width, label='Logistic Regression', color='#ff6584', alpha=0.85)

ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../data/plot10_model_compare.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Importance (Random Forest) ──
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 5))
colors_fi = ['#6c63ff' if x > 0.2 else '#43e97b' for x in feature_importance['Importance']]
plt.barh(feature_importance['Feature'], feature_importance['Importance'],
         color=colors_fi, edgecolor='white')
plt.title('Random Forest — Feature Importance', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
for i, (val, name) in enumerate(zip(feature_importance['Importance'], feature_importance['Feature'])):
    plt.text(val + 0.002, i, f'{val:.4f}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plot11_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Most important feature for predicting job role is:', 
      feature_importance.iloc[-1]['Feature'])

In [ ]:
# ── Hyperparameter Tuning for Random Forest ──
print('=== HYPERPARAMETER TUNING ===')

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_train, y_train_role)

print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV Score: {grid_search.best_score_:.4f}')

# Retrain with best params
best_rf = grid_search.best_estimator_
best_rf_pred = best_rf.predict(X_test)
best_rf_acc = accuracy_score(y_test_role, best_rf_pred)
print(f'\nTuned RF Test Accuracy: {best_rf_acc:.4f} ({best_rf_acc*100:.2f}%)')
print(f'Improvement over baseline: +{(best_rf_acc - rf_accuracy)*100:.2f}%')

## Step 10 — Insights & Conclusion

In [ ]:
# ── Final Summary ──
print('=' * 55)
print('       FINAL PROJECT SUMMARY & INSIGHTS')
print('=' * 55)

best_model = comparison_df.loc[comparison_df['Accuracy'].str.replace('%','').astype(float).idxmax(), 'Model']

print(f"""
📊 DATASET INSIGHTS:
  • Total jobs analyzed: {len(df)}
  • Most in-demand skill: {skill_counts[0][0]}
  • Average skills per job: {df['skill_count'].mean():.1f}

🤖 MODEL PERFORMANCE:
  • KNN Accuracy:           {knn_accuracy*100:.2f}%
  • Random Forest Accuracy: {rf_accuracy*100:.2f}%
  • Logistic Regression:    {lr_accuracy*100:.2f}%
  • Best Model: {best_model}
  • After Hyperparameter Tuning: {best_rf_acc*100:.2f}%

🎯 KEY FINDINGS:
  • keyword_match is the strongest predictor of job fit
  • Data Science roles require more skills than Web Dev
  • Random Forest outperforms KNN due to ensemble learning
  • Hyperparameter tuning improved accuracy

📈 FUTURE IMPROVEMENTS:
  • Use larger real-world dataset (Kaggle)
  • Add NLP for better text understanding  
  • Implement BERT embeddings for semantic matching
  • Add more job categories
""")
print('=' * 55)

In [ ]:
# ── Save Final Models ──
import pickle
import os

os.makedirs('../models', exist_ok=True)

with open('../models/knn_final.pkl', 'wb') as f:
    pickle.dump(knn_model, f)

with open('../models/rf_final.pkl', 'wb') as f:
    pickle.dump(best_rf, f)

with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../models/label_encoder_role.pkl', 'wb') as f:
    pickle.dump(le_role, f)

print('✅ All models saved successfully!')
print('Files saved:')
print('  • models/knn_final.pkl')
print('  • models/rf_final.pkl')
print('  • models/scaler.pkl')
print('  • models/label_encoder_role.pkl')